# Project 3 — Small-Model Specialisation
### QLoRA fine-tune + a three-tier evaluation harness
**PEFT · TRL · LLM-as-judge · Cohen's kappa · McNemar's test · eval CI gate**

---

## How to use this notebook

1. **Runtime → Change runtime type → T4 GPU.**
2. Run top to bottom. Training takes ~10–20 minutes on a free T4.
3. Theory comes first in each section; the code proves it.

## The thesis

> **A 0.5B model fine-tuned on your task can beat a 70B general model on that task — but only if you can *prove* it. The proof is harder than the training.**

Anyone can run `SFTTrainer`. What separates an engineer from a notebook-runner is the **evaluation harness**: knowing that an LLM judge is a *measurement instrument* that must itself be calibrated, and that comparing two models on the same test set requires a **paired** statistical test.

## Section map

| # | Section | What you learn |
|---|---------|----------------|
| 1 | Why specialise | The economics and when it's the wrong call |
| 2 | Quantization: NF4 | Why 4-bit normal-float, double quant, paged optimizers |
| 3 | LoRA math | Rank decomposition, r, alpha, target modules — with a numpy demo |
| 4 | QLoRA = the combination | How gradients flow through a frozen 4-bit base |
| 5 | Task + data | AG News classification, why a verifiable task |
| 6 | **Baseline first** | The rule people break most often |
| 7 | Chat templates & loss masking | Completion-only training |
| 8 | Training | Every hyperparameter explained |
| 9 | Post-train eval | Same harness, same seed |
| 10 | **Tier 1** — deterministic | Accuracy, macro-F1, confusion matrix |
| 11 | **Tier 2** — LLM-as-judge | Rubrics, position bias, pointwise vs pairwise |
| 12 | **Tier 3** — Cohen's kappa | Validating the judge against humans |
| 13 | McNemar's test | The right test for paired model comparison |
| 14 | Bootstrap CIs | Uncertainty on your headline number |
| 15 | Eval CI gate | The script + GitHub Actions workflow |
| 16 | Failure modes | Contamination, judge drift, overfit-to-eval |
| 17 | Interview cheat sheet | |


---
# 1. Why specialise a small model

### The economics
A 0.5B model is roughly **140× cheaper to serve** than a 70B (and by Project 2's concurrency model, much more than 140× in throughput terms on a constrained GPU). If your task is narrow — classification, extraction, routing, a fixed-format transformation — a large general model is enormous overkill.

### What fine-tuning actually buys you
| It DOES fix | It does NOT fix |
|---|---|
| Output **format** compliance | Missing **knowledge** (use RAG) |
| Consistent **style / tone** | **Reasoning** ability beyond model capacity |
| **Task-specific** behaviour | Facts that change (use retrieval) |
| Removing verbose preambles | Fundamental capability ceilings |

> **The single most useful heuristic:** *Fine-tuning teaches form. Retrieval supplies facts.* If your errors are "wrong format / wrong style / too chatty", fine-tune. If they're "didn't know / made it up", that's RAG, not SFT.

### The order you should try things (and say this in an interview)
1. Better prompt + few-shot examples
2. RAG
3. A bigger model
4. **Then** fine-tune a small model
5. Distill from the big model into the small one

Fine-tuning first is a common junior mistake — it's the most expensive option and the hardest to maintain.

### Distillation angle
The strongest practical recipe: use a large model to **generate training labels/rationales**, then fine-tune a small model on them. You get most of the big model's task behaviour at small-model cost. That's exactly what this notebook does in miniature.


---
# 2. Quantization for training — NF4, double quant, paged optimizers

QLoRA (Dettmers et al., 2023) has **three** components. Most people can only name the first.

### (a) 4-bit NormalFloat (NF4)

Standard INT4 spaces its 16 levels **uniformly**. But neural network weights are approximately **zero-centred normal**. Uniform levels waste most of their resolution on the tails where almost no weights live.

NF4 is an **information-theoretically optimal** datatype for normally-distributed data: it places its 16 quantization levels at the **quantiles** of a standard normal, so each level holds roughly the same number of weights. Equal-probability bins = maximum information per bit.

It is also **exactly zero-symmetric** — one level is exactly 0.0, which matters because zeros are common and asymmetric error around zero biases the whole layer.

Weights are quantized in **blocks of 64**, each with its own scaling constant (the block's absmax), so a single outlier only corrupts its own block.

### (b) Double Quantization

Those per-block constants cost memory too: one fp32 per 64 weights = `32/64` = **0.5 bits per parameter**. DQ quantizes *the constants themselves* to 8-bit (in blocks of 256), cutting that to ~0.127 bits/param. Saves ~0.37 bits/param — about **3 GB on a 65B model**. Small idea, real money.

### (c) Paged Optimizers

Gradient checkpointing causes memory **spikes** when a long sequence arrives. Paged optimizers use NVIDIA unified memory to automatically **page optimizer state out to CPU RAM** during a spike and back in after — like OS swap. Turns a hard OOM crash into a slowdown.

### Critical detail: compute dtype ≠ storage dtype

Weights are **stored** in NF4. During the forward pass each block is **dequantized to fp16/bf16 on the fly** to do the matmul, then discarded. So:
- Memory saving: real and large.
- Compute saving: **none** — QLoRA is actually a bit *slower* per step than fp16 LoRA. You trade time for memory.

This trips people up constantly. Say it clearly and you look like you've actually done it.

> On T4 use `compute_dtype=torch.float16` (T4 is sm75, **no bf16**). On A100+ use `bfloat16` — wider exponent range, far fewer overflow/NaN issues.


In [ ]:
# Install. Colab often has some of these; pin loosely.
!pip -q install -U "transformers>=4.44" "peft>=0.12" "trl>=0.11" "bitsandbytes>=0.43" \
                   "accelerate>=0.33" datasets scikit-learn statsmodels 2>/dev/null | tail -2

import torch, numpy as np, random, os, json, warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
# Reproducibility matters enormously for an eval harness: if your metric moves
# because of a seed, your CI gate is noise and will block good PRs.

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, "| %.1f GB" % (p.total_memory/1e9), "| sm%d%d" % (p.major, p.minor))
    BF16_OK = p.major >= 8
    print("bf16 supported:", BF16_OK, "-> compute dtype will be", "bfloat16" if BF16_OK else "float16")


In [ ]:
# --- Visualise WHY NF4 beats INT4 for normally-distributed weights ---
import matplotlib.pyplot as plt
from scipy import stats

w = np.random.randn(200000)          # stand-in for a real weight tensor
w = w / np.abs(w).max()              # absmax normalize, as bitsandbytes does

# INT4: 16 uniformly spaced levels
int4_levels = np.linspace(-1, 1, 16)

# NF4: 16 levels at the quantiles of a normal, rescaled to [-1,1]
qs = np.linspace(0.5/16, 1-0.5/16, 16)
nf4_levels = stats.norm.ppf(qs)
nf4_levels = nf4_levels / np.abs(nf4_levels).max()

def quantize(x, levels):
    idx = np.abs(x[:,None] - levels[None,:]).argmin(1)
    return levels[idx]

for name, lv in [("INT4 (uniform)", int4_levels), ("NF4 (quantile)", nf4_levels)]:
    q = quantize(w, lv)
    mse = ((w-q)**2).mean()
    print("%-18s MSE = %.3e" % (name, mse))

fig, ax = plt.subplots(figsize=(11,4))
ax.hist(w, bins=200, density=True, alpha=.3, color="gray", label="weight distribution")
ax.vlines(int4_levels, 0, 1.6, color="tab:red",   lw=1.6, label="INT4 levels (uniform)")
ax.vlines(nf4_levels,  0, 1.2, color="tab:blue",  lw=1.6, ls="--", label="NF4 levels (quantile)")
ax.set_title("NF4 puts its levels where the weights actually are")
ax.legend(); ax.set_xlabel("normalized weight value"); plt.tight_layout(); plt.show()

print("""
Notice NF4 clusters levels near zero, where the mass is, and spreads them out
in the tails where few weights live. Equal-probability bins = each of the 16
codes carries the same information = minimum expected quantization error.""")


---
# 3. LoRA — the math

### The hypothesis
Fine-tuning updates a weight matrix `W₀` to `W₀ + ΔW`. LoRA's claim: **`ΔW` has low intrinsic rank.** Adaptation to a narrow task doesn't need a full-rank update — it lives in a small subspace.

$$ \Delta W = BA, \qquad B \in \mathbb{R}^{d \times r},\ A \in \mathbb{R}^{r \times k},\ r \ll \min(d,k) $$

The forward pass becomes:

$$ h = W_0 x + \frac{\alpha}{r} BA x $$

- `W₀` is **frozen** (and in QLoRA, 4-bit).
- Only `A` and `B` are trained.
- Parameters: `r(d+k)` instead of `dk`. For `d=k=4096, r=16`: **131K instead of 16.7M** — 0.8%.

### Initialization is not arbitrary
`A ~ N(0, σ²)` and **`B = 0`**. So `BA = 0` at step 0, meaning the model starts **exactly equal to the pretrained model**. Training begins from a known-good point rather than a random perturbation. If both were random, you'd inject noise into every layer at initialization and destroy the pretrained behaviour.

### The `alpha/r` scaling
`α` is a scaling constant. The update is scaled by `α/r`, which **decouples the learning rate from the rank**: raise `r` and the per-parameter contribution shrinks proportionally, so you don't have to retune LR. Common convention: `α = 2r` (e.g. `r=16, α=32`).

### Which modules to target
- **Attention only** (`q_proj, k_proj, v_proj, o_proj`) — the original paper; cheap.
- **All linear layers** (add `gate_proj, up_proj, down_proj`) — the QLoRA paper found this matters **a lot**, and it's now the default recommendation. The MLP holds most of the parameters and much of the task-specific behaviour.

### Choosing `r`
- `r = 4–8`: style, tone, format compliance
- `r = 16–32`: typical task adaptation ← start here
- `r = 64–128`: substantial new behaviour, larger datasets
- Higher `r` isn't reliably better; it mostly increases overfitting risk on small data.

### The deployment advantage nobody mentions
LoRA adapters can be **merged** into the base weights (`W = W₀ + (α/r)BA`) giving **zero inference overhead** — unlike adapter layers or prompt tuning, which add latency forever. Or keep them separate and **hot-swap** many adapters over one shared base (this is what multi-tenant LoRA serving, e.g. S-LoRA / vLLM's LoRA support, does).

> **Caveat for QLoRA specifically:** merging a LoRA into a *4-bit* base requires dequantizing to fp16 first — you cannot merge into NF4 losslessly. In practice: reload the base in fp16, merge, then re-quantize (e.g. to AWQ) for serving.


In [ ]:
# LoRA parameter arithmetic + a demo that a low-rank update can carry real signal
def lora_params(d, k, r): return r*(d+k)

print("%-10s %14s %14s %10s" % ("r","full ΔW","LoRA params","% of full"))
print("-"*52)
d = k = 4096
for r in [1,2,4,8,16,32,64,128,256]:
    lp = lora_params(d,k,r)
    print("%-10d %14s %14s %9.2f%%" % (r, f"{d*k:,}", f"{lp:,}", 100*lp/(d*k)))

# --- Do real fine-tuning updates actually have low rank? Empirical check. ---
np.random.seed(0)
d = 256
W0 = np.random.randn(d,d)/np.sqrt(d)
# Simulate a "task update" that only needs a few directions
true_r = 8
dW = (np.random.randn(d,true_r) @ np.random.randn(true_r,d)) * 0.05
dW += np.random.randn(d,d)*0.002          # plus a little full-rank noise

s = np.linalg.svd(dW, compute_uv=False)
energy = np.cumsum(s**2)/np.sum(s**2)

fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].semilogy(s[:64], "o-"); ax[0].set_title("Singular values of ΔW"); ax[0].grid(alpha=.3)
ax[0].set_xlabel("index")
ax[1].plot(energy[:64], "o-"); ax[1].axhline(.95, ls="--", color="r")
ax[1].set_title("Cumulative energy captured by rank r"); ax[1].set_xlabel("rank r"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

r95 = int(np.searchsorted(energy, 0.95))+1
print("Rank needed to capture 95%% of the update: r = %d  (out of %d)" % (r95, d))
print("That is the LoRA hypothesis in one number.")


---
# 4. QLoRA = NF4 base + LoRA adapters

### How gradients flow through a frozen 4-bit model

This is the question that separates people who read the paper from people who skimmed a blog post.

1. **Forward:** each NF4 weight block is dequantized to fp16 on the fly → matmul → result. The dequantized copy is transient.
2. **Backward:** gradients flow **through** the frozen base weights (you need `∂L/∂x` to reach earlier layers) but **no gradient is stored for `W₀`** — it has `requires_grad=False`.
3. Only `A` and `B` accumulate gradients and have optimizer state.

So the memory you save is: base weight storage (4× less) **plus** — and this is the bigger win — **no gradients and no Adam optimizer state for the base.** Adam keeps 2 fp32 moments per trainable parameter: full fine-tuning of a 7B model needs ~14 GB weights + 14 GB grads + 56 GB optimizer ≈ 84 GB. QLoRA needs ~3.5 GB + a few hundred MB. That's how a 65B fits on one 48 GB card.

### The three memory consumers, ranked
| Consumer | Full FT (7B) | QLoRA (7B) |
|---|---|---|
| Base weights | 14 GB (fp16) | 3.5 GB (NF4) |
| Gradients | 14 GB | ~0.01 GB |
| Adam state (2× fp32) | 56 GB | ~0.05 GB |
| Activations | varies | reduced by grad checkpointing |

### `prepare_model_for_kbit_training` — what it actually does
- Casts layernorms and the LM head to fp32 (they're numerically sensitive; fp16 layernorms produce NaNs).
- Enables **gradient checkpointing** (recompute activations in backward instead of storing them — trades ~30% compute for large activation memory savings).
- Makes the input embeddings require grad so checkpointing works correctly.

### Gradient checkpointing, briefly
Normally you store every layer's activations for the backward pass — that's O(L) memory. Checkpointing stores only a few and **recomputes** the rest during backward. O(√L) memory, ~1.3× compute. Essential on a T4.


---
# 5. The task and the data

We use **AG News topic classification** (World / Sports / Business / Sci-Tech).

### Why a classification task for a fine-tuning demo?
Because **every tier of the eval harness needs something different**, and this task gives all three cleanly:

- **Tier 1** needs a *verifiable* answer → the gold label. Exact match works.
- **Tier 2** needs *open-ended* output to judge → we also ask for a one-sentence rationale.
- **Tier 3** needs **paired binary outcomes** (correct/incorrect per item, per model) → exactly what McNemar's test consumes.

A pure chat/summarisation task would give you no ground truth, and then you'd have nothing to calibrate the judge against. **Design the task so the eval is possible.** That's a real engineering decision, and worth saying out loud.

### What we're actually testing
A base 0.5B instruct model will get this task partly right but will be **verbose, inconsistent in format, and prone to inventing categories**. Fine-tuning should fix format and boost accuracy. That's the classic "fine-tuning teaches form" result.


In [ ]:
from datasets import load_dataset

LABELS = ["World", "Sports", "Business", "Sci/Tech"]

def build_data(n_train=800, n_test=200):
    try:
        ds = load_dataset("ag_news")
        tr = ds["train"].shuffle(seed=SEED).select(range(n_train))
        te = ds["test"].shuffle(seed=SEED).select(range(n_test))
        train = [{"text": r["text"][:400], "label": LABELS[r["label"]]} for r in tr]
        test  = [{"text": r["text"][:400], "label": LABELS[r["label"]]} for r in te]
        return train, test
    except Exception as e:
        print("Download failed (%s). Using synthetic fallback." % str(e)[:60])
        import random as _r
        seeds = {
            "World":    ["UN envoy meets officials in {} over ceasefire talks",
                         "Elections in {} draw record turnout amid protests",
                         "Floods displace thousands across northern {}"],
            "Sports":   ["{} clinch title after dramatic penalty shootout",
                         "Star striker signs record deal with {} United",
                         "{} sets new world record in the 200m final"],
            "Business": ["{} Corp shares slide after weak quarterly guidance",
                         "Central bank of {} holds interest rates steady",
                         "{} announces merger in $4bn all-stock deal"],
            "Sci/Tech": ["Researchers at {} unveil new battery chemistry",
                         "{} launches satellite constellation for broadband",
                         "New model from {} lab tops language benchmarks"],
        }
        places = ["Kenya","Brazil","Norway","Vietnam","Chile","Poland","Ghana","Nepal",
                  "Mexico","Sweden","Egypt","Peru","Latvia","Oman","Fiji","Bolivia"]
        rows=[]
        for lab, temps in seeds.items():
            for t in temps:
                for p in places:
                    rows.append({"text": t.format(p), "label": lab})
        _r.Random(SEED).shuffle(rows)
        return rows[:n_train], rows[n_train:n_train+n_test]

train_rows, test_rows = build_data()
print("train:", len(train_rows), "| test:", len(test_rows))
from collections import Counter
print("train label balance:", Counter(r["label"] for r in train_rows))
print("test  label balance:", Counter(r["label"] for r in test_rows))
print("\nExample:\n ", train_rows[0]["text"][:180], "\n  ->", train_rows[0]["label"])


---
# 6. Baseline FIRST — the rule people break most often

> **Never train before you have measured the baseline with the exact same harness, the exact same prompt, and the exact same seed.**

Reasons this is non-negotiable:
1. Without it you cannot claim an improvement — only a number.
2. The baseline often surprises you. If a good prompt already gets 88%, fine-tuning for 2% is not worth the maintenance burden, and you should find that out **before** spending the GPU hours.
3. McNemar's test (Section 13) is **paired** — it needs per-item results from both models on the *same items in the same order*. If you don't save baseline predictions per item, you cannot run it later.

We therefore save predictions **per example**, not just aggregate scores. Aggregate-only logging is the mistake that makes proper statistics impossible after the fact.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

BASE_ID = "Qwen/Qwen2.5-0.5B-Instruct"
COMPUTE_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_properties(0).major>=8) else torch.float16

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # <- the NormalFloat datatype from Section 2
    bnb_4bit_use_double_quant=True,        # <- double quantization
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # <- dequantize to THIS for the matmul
)

tok = AutoTokenizer.from_pretrained(BASE_ID)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "left"                  # left padding is REQUIRED for batched generation

model = AutoModelForCausalLM.from_pretrained(
    BASE_ID, quantization_config=bnb_cfg, device_map="cuda", torch_dtype=COMPUTE_DTYPE)
model.config.use_cache = True
model.eval()

print("loaded 4-bit. VRAM used: %.2f GB" % (torch.cuda.memory_allocated()/1e9))
fp16_would_be = sum(p.numel() for p in model.parameters())*2/1e9
print("(fp16 would be roughly %.2f GB of weights)" % fp16_would_be)


In [ ]:
SYSTEM = ("You are a news topic classifier. Classify the article into exactly one of: "
          "World, Sports, Business, Sci/Tech.\n"
          "Reply in exactly this format and nothing else:\n"
          "Label: <one of World|Sports|Business|Sci/Tech>\n"
          "Reason: <one short sentence>")

def build_prompt(text):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":"Article: " + text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def target_text(label, reason):
    return "Label: %s\nReason: %s" % (label, reason)

import re
def parse(out):
    """Robust parsing. Returns (label_or_None, reason). NEVER silently guess."""
    lab, reason = None, ""
    m = re.search(r"Label:\s*([A-Za-z/ ]+)", out)
    if m:
        cand = m.group(1).strip().rstrip(".").strip()
        for L in LABELS:
            if cand.lower().startswith(L.lower()[:4]): lab = L; break
    if lab is None:      # fall back to bare mention anywhere in the output
        for L in LABELS:
            if re.search(r"\b"+re.escape(L)+r"\b", out, re.I): lab = L; break
    m = re.search(r"Reason:\s*(.+)", out, re.S)
    if m: reason = m.group(1).strip().split("\n")[0][:300]
    return lab, reason

print(build_prompt("Apple reports record quarterly revenue.")[:500])


In [ ]:
@torch.inference_mode()
def run_eval(mdl, rows, batch=8, max_new=48, tag=""):
    """Returns per-item records. Per-item is essential: McNemar needs pairs."""
    recs = []
    for i in range(0, len(rows), batch):
        chunk = rows[i:i+batch]
        prompts = [build_prompt(r["text"]) for r in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, truncation=True,
                  max_length=512).to("cuda")
        out = mdl.generate(**enc, max_new_tokens=max_new, do_sample=False,   # greedy = reproducible
                           pad_token_id=tok.pad_token_id)
        gen = out[:, enc["input_ids"].shape[1]:]
        texts = tok.batch_decode(gen, skip_special_tokens=True)
        for r, t in zip(chunk, texts):
            lab, reason = parse(t)
            recs.append({"text": r["text"], "gold": r["label"], "pred": lab,
                         "reason": reason, "raw": t.strip(),
                         "correct": int(lab == r["label"]),
                         "format_ok": int(t.strip().startswith("Label:"))})
        if (i//batch) % 5 == 0:
            print("  %s %d/%d" % (tag, min(i+batch,len(rows)), len(rows)), end="\r")
    print()
    return recs

print("Evaluating BASE model (before any training)...")
base_recs = run_eval(model, test_rows, tag="base")

acc = np.mean([r["correct"] for r in base_recs])
fmt = np.mean([r["format_ok"] for r in base_recs])
unparsed = np.mean([r["pred"] is None for r in base_recs])
print("\nBASE  accuracy       : %.3f" % acc)
print("BASE  format compliance: %.3f" % fmt)
print("BASE  unparseable      : %.3f" % unparsed)
print("\n--- three raw outputs (look at how chatty/inconsistent it is) ---")
for r in base_recs[:3]:
    print("gold=%-9s pred=%-9s | %s" % (r["gold"], r["pred"], r["raw"][:120].replace("\n"," / ")))


---
# 7. Chat templates and completion-only loss masking

### Chat templates
Every instruct model was trained with specific control tokens (`<|im_start|>`, `[INST]`, etc.). `apply_chat_template` reads the template stored in the tokenizer config and formats correctly. **If you fine-tune with a different format than the model was instruction-tuned with, you fight the pretrained behaviour instead of building on it.** A very common silent bug.

### Completion-only loss (a.k.a. prompt masking)

By default, causal LM training computes loss on **every** token — including the prompt. That means the model spends capacity learning to *predict your own system prompt*, which is useless.

Set the label of every prompt token to `-100` (PyTorch's ignore index in `CrossEntropyLoss`) so gradient flows **only from the completion**.

TRL's `DataCollatorForCompletionOnlyLM` does this: you give it a `response_template` string, it finds that token sequence in each example and masks everything before it.

**When does it matter?** Most on short completions with long prompts — exactly our case (a long article prompt, an 8-token answer). Without masking, >90% of the loss signal is about the article text, and your effective learning rate on the thing you care about is tiny.

### Packing
Concatenating multiple short examples into one `max_seq_length` sequence to avoid wasting compute on padding. Big throughput win, but it can leak context across examples. Use it for pretraining-style data; be careful for instruction data. We disable it here for clarity.


In [ ]:
# Build supervised examples. The target includes the rationale so we ALSO train
# the free-text behaviour that Tier 2 will judge.
REASONS = {
 "World":    "It concerns international affairs, politics or society.",
 "Sports":   "It concerns athletes, teams or sporting competition.",
 "Business": "It concerns companies, markets, or the economy.",
 "Sci/Tech": "It concerns science, technology or research.",
}

def to_example(r):
    return {"text": build_prompt(r["text"]) + target_text(r["label"], REASONS[r["label"]]) + tok.eos_token}

from datasets import Dataset
train_ds = Dataset.from_list([to_example(r) for r in train_rows])
print(train_ds[0]["text"][-400:])
print("\n^ note: prompt + completion in ONE string. The collator masks the prompt part.")


---
# 8. Training — every hyperparameter, and why

| Param | Value | Why |
|---|---|---|
| `r` | 16 | Task adaptation range; 0.8% of params |
| `lora_alpha` | 32 | `α = 2r` convention; scaling is `α/r = 2` |
| `lora_dropout` | 0.05 | Regularizes the adapter on a small dataset |
| `target_modules` | all linear | QLoRA paper: including MLP matters a lot |
| `learning_rate` | 2e-4 | **~10–100× higher than full FT.** You're training few, freshly-initialized params — they need a big LR. Using 2e-5 here is the classic "why isn't it learning" bug |
| `lr_scheduler` | cosine | Smooth decay; standard for short SFT runs |
| `warmup_ratio` | 0.03 | Adam's moment estimates are garbage for the first steps; warmup avoids an early destructive update |
| `optim` | `paged_adamw_8bit` | 8-bit Adam state + paging (Section 2c) |
| `per_device_batch` × `grad_accum` | 4 × 4 = 16 | **Effective** batch is what matters. Accumulation buys large-batch stability on small VRAM at the cost of wall time |
| `max_grad_norm` | 0.3 | Clipping; fp16 + small batch can produce spikes |
| `num_epochs` | 2 | Small data overfits fast. Watch eval loss, not train loss |
| `fp16` / `bf16` | per-GPU | T4 → fp16; A100+ → bf16 (fewer NaNs) |

### The two failure modes and their signatures
- **Train loss falls, eval loss rises** → overfitting. Fewer epochs, more dropout, more data, lower `r`.
- **Loss is flat** → LR too low (most likely), wrong chat template, or LoRA not attached (check `print_trainable_parameters()` — if it says 0, your `target_modules` names are wrong for this architecture).

### Catastrophic forgetting
Fine-tuning narrowly degrades general ability. LoRA is *less* prone than full FT because the base is frozen and the update is low-rank — but it still happens. Mitigations: mix ~5–10% general instruction data into your training set, keep `r` modest, fewer epochs, and **always evaluate on a held-out general benchmark**, not just your task.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
# ^ If this prints 0 trainable params, your target_modules names don't match
#   this architecture. Run:  print([n for n,_ in model.named_modules()][:60])


In [ ]:
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from transformers import TrainingArguments

USE_BF16 = torch.cuda.get_device_properties(0).major >= 8

# The response template: everything BEFORE this in each example gets label -100.
RESPONSE_TEMPLATE = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(response_template=RESPONSE_TEMPLATE, tokenizer=tok)

try:
    cfg = SFTConfig(
        output_dir="./qlora-agnews",
        num_train_epochs=2,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,        # effective batch = 16
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        max_grad_norm=0.3,
        optim="paged_adamw_8bit",
        logging_steps=10,
        save_strategy="no",
        fp16=not USE_BF16, bf16=USE_BF16,
        max_seq_length=512,
        packing=False,
        dataset_text_field="text",
        report_to="none",
        seed=SEED,
    )
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, data_collator=collator)
except TypeError:
    # Older TRL: args split between TrainingArguments and SFTTrainer kwargs
    args = TrainingArguments(
        output_dir="./qlora-agnews", num_train_epochs=2,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
        max_grad_norm=0.3, optim="paged_adamw_8bit", logging_steps=10,
        save_strategy="no", fp16=not USE_BF16, bf16=USE_BF16, report_to="none", seed=SEED)
    trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                         dataset_text_field="text", max_seq_length=512,
                         data_collator=collator, packing=False)

print("Training... (~10-20 min on a T4)")
res = trainer.train()
print("\nfinal train loss: %.4f" % res.training_loss)


In [ ]:
# Plot the loss curve. A curve that plateaus early with more epochs left is
# your cue to cut epochs; a curve still descending means you can train longer.
hist = [h for h in trainer.state.log_history if "loss" in h]
if hist:
    plt.figure(figsize=(8,3.5))
    plt.plot([h["step"] for h in hist], [h["loss"] for h in hist], "o-")
    plt.xlabel("step"); plt.ylabel("train loss"); plt.title("QLoRA training")
    plt.grid(alpha=.3); plt.tight_layout(); plt.show()

model.eval()
model.config.use_cache = True
trainer.model.save_pretrained("./qlora-adapter")   # adapter only: a few MB, not GB
import os
sz = sum(os.path.getsize(os.path.join("./qlora-adapter",f))
         for f in os.listdir("./qlora-adapter"))/1e6
print("adapter size on disk: %.1f MB  <- this is the whole artifact you ship" % sz)


---
# 9. Post-training evaluation — identical harness

Same prompt, same seed, same greedy decoding, same parser, same test set in the same order. **Any difference in the harness invalidates the comparison.**

This is why the eval code was written as a reusable function *before* training rather than improvised afterwards.


In [ ]:
print("Evaluating TUNED model...")
tuned_recs = run_eval(model, test_rows, tag="tuned")

def summarize(recs, name):
    return {
        "name": name,
        "accuracy": float(np.mean([r["correct"] for r in recs])),
        "format_ok": float(np.mean([r["format_ok"] for r in recs])),
        "unparseable": float(np.mean([r["pred"] is None for r in recs])),
        "avg_len": float(np.mean([len(r["raw"]) for r in recs])),
    }

sb, st = summarize(base_recs,"base"), summarize(tuned_recs,"tuned")
print("\n%-16s %10s %10s %10s" % ("metric","base","tuned","delta"))
print("-"*50)
for k in ["accuracy","format_ok","unparseable","avg_len"]:
    print("%-16s %10.3f %10.3f %+10.3f" % (k, sb[k], st[k], st[k]-sb[k]))

print("\nLook at avg_len: fine-tuning usually collapses output length dramatically.")
print("That alone is a real production win -- fewer output tokens = lower cost")
print("and lower latency (Project 2: total latency ~ TTFT + n_tokens * TPOT).")


---
# 10. TIER 1 — Deterministic metrics

**Cheap, reproducible, unambiguous. Run on every commit.**

### Metrics and when each is wrong
- **Accuracy** — meaningless on imbalanced data. 95% accuracy on a 95/5 split = predicting the majority class.
- **Precision** = TP/(TP+FP): "when I said yes, was I right?" Optimize when false positives are expensive (spam filters).
- **Recall** = TP/(TP+FN): "of all the real yeses, how many did I catch?" Optimize when false negatives are expensive (disease screening).
- **F1** = harmonic mean. Harmonic, not arithmetic, because it punishes imbalance — precision 1.0 / recall 0.0 gives F1 = 0, not 0.5.
- **Macro-F1** = unweighted mean over classes → every class counts equally, so rare classes can't be ignored. **Micro-F1** = global counts → equals accuracy in single-label multiclass. **Report macro** when classes are imbalanced.

### Task-specific metrics you should always add
Beyond accuracy: **format compliance rate**, **unparseable rate**, **output length**. In production these often matter more than the accuracy delta — a model that's 1% more accurate but breaks your JSON parser 5% of the time is a net loss.


In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

def tier1(recs):
    y_true = [r["gold"] for r in recs]
    y_pred = [r["pred"] if r["pred"] is not None else "UNPARSED" for r in recs]
    labs = LABELS + ["UNPARSED"]
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, labels=LABELS, average="micro", zero_division=0),
        "y_true": y_true, "y_pred": y_pred, "labels": labs,
    }

t1b, t1t = tier1(base_recs), tier1(tuned_recs)
print("%-12s %10s %10s %+10s" % ("metric","base","tuned","delta"))
print("-"*46)
for k in ["accuracy","macro_f1","micro_f1"]:
    print("%-12s %10.4f %10.4f %+10.4f" % (k, t1b[k], t1t[k], t1t[k]-t1b[k]))

print("\n=== TUNED per-class report ===")
print(classification_report(t1t["y_true"], t1t["y_pred"], labels=LABELS, zero_division=0))

fig, ax = plt.subplots(1,2, figsize=(13,5))
for a, t, name in [(ax[0],t1b,"BASE"), (ax[1],t1t,"TUNED")]:
    cm = confusion_matrix(t["y_true"], t["y_pred"], labels=t["labels"])
    im = a.imshow(cm, cmap="Blues")
    a.set_xticks(range(len(t["labels"]))); a.set_xticklabels(t["labels"], rotation=45, ha="right")
    a.set_yticks(range(len(t["labels"]))); a.set_yticklabels(t["labels"])
    a.set_title("%s (acc %.3f)" % (name, t["accuracy"]))
    a.set_xlabel("predicted"); a.set_ylabel("gold")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if cm[i,j]: a.text(j,i,cm[i,j],ha="center",va="center",
                               color="white" if cm[i,j]>cm.max()/2 else "black")
plt.tight_layout(); plt.show()
print("Read the OFF-DIAGONAL cells. Which classes get confused tells you what the")
print("model actually misunderstands -- an aggregate accuracy number never does.")


---
# 11. TIER 2 — LLM-as-judge

Tier 1 can't score a free-text rationale. LLM-as-judge can, and it scales. But:

> **An LLM judge is a measurement instrument. Instruments must be calibrated. An uncalibrated judge produces numbers with unknown validity — which is worse than no numbers, because they look authoritative.**

### Known biases (name these and you sound like you've deployed one)
| Bias | What it is | Mitigation |
|---|---|---|
| **Position** | Prefers whichever answer is shown first (or last) | Run both orders, average; report the swap rate |
| **Verbosity** | Prefers longer answers regardless of quality | Length-control the rubric; explicitly penalise padding |
| **Self-preference** | Prefers text from its own model family | Use a different family as judge |
| **Style over substance** | Confident, well-formatted wrong answers score high | Force the judge to check factual criteria first |
| **Leniency / scale compression** | Everything gets 4/5 | Anchored rubric with concrete descriptors per point |

### Design rules for a judge prompt
1. **Explicit rubric with anchors.** Not "rate 1–5" — define what a 1, 3, and 5 look like.
2. **Reason before scoring.** Ask for a short justification *then* the score. Scoring first makes the reason post-hoc rationalization.
3. **Structured output** (JSON) so parsing is deterministic.
4. **Reference-based when possible.** Giving the gold answer turns a hard generative judgement into an easier comparison and sharply raises agreement with humans.
5. **Temperature 0** for reproducibility.

### Pointwise vs Pairwise
- **Pointwise**: score each output independently. Absolute scale, comparable across runs, but suffers scale drift.
- **Pairwise**: "A or B?" Much higher human agreement (relative judgements are easier), but O(n²) and gives no absolute number. Use pairwise for model-vs-model, pointwise for tracking over time.

### A crucial practical point
For small tasks, the judge should be **stronger than the model under test**. Judging with the same 0.5B model you're evaluating is close to meaningless. Below we implement two backends: a **local judge** (works offline, illustrates mechanics) and an **API judge** (what you'd use in production).


In [ ]:
JUDGE_RUBRIC = """You are an impartial evaluator. Score the RATIONALE for a news topic classification.

Criteria:
5 = Rationale is accurate, specific to this article, and clearly supports the correct topic.
4 = Accurate and supports the topic, but generic.
3 = Plausible but vague, or only loosely tied to the article.
2 = Largely irrelevant, or supports the wrong topic.
1 = Incoherent, empty, or contradicts the article.

Return ONLY a JSON object, no markdown fences, no extra text:
{"reasoning": "<one short sentence>", "score": <integer 1-5>}"""

def judge_prompt(article, gold, pred, reason):
    return [
        {"role":"system","content":JUDGE_RUBRIC},
        {"role":"user","content":
            "ARTICLE: %s\n\nGOLD TOPIC: %s\nMODEL PREDICTED: %s\nMODEL RATIONALE: %s\n\nScore the rationale."
            % (article[:300], gold, pred, reason if reason else "(none provided)")}
    ]

def parse_judge(txt):
    m = re.search(r"\{.*?\}", txt, re.S)
    if m:
        try:
            o = json.loads(m.group(0))
            s = int(o.get("score", 0))
            if 1 <= s <= 5: return s, str(o.get("reasoning",""))[:200]
        except Exception: pass
    m = re.search(r"[1-5]", txt)          # last-resort fallback
    return (int(m.group(0)), "fallback-parse") if m else (None, "unparsed")
print("judge prompt defined")


In [ ]:
# ---- Judge backend A: LOCAL (works offline; illustrates the mechanics) ------
from transformers import AutoModelForCausalLM as AMLM
JUDGE_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # stronger than the 0.5B under test

USE_LOCAL_JUDGE = True
judge_model = judge_tok = None
if USE_LOCAL_JUDGE:
    try:
        judge_tok = AutoTokenizer.from_pretrained(JUDGE_ID)
        if judge_tok.pad_token is None: judge_tok.pad_token = judge_tok.eos_token
        judge_tok.padding_side = "left"
        judge_model = AMLM.from_pretrained(JUDGE_ID, quantization_config=bnb_cfg,
                                           device_map="cuda", torch_dtype=COMPUTE_DTYPE)
        judge_model.eval()
        print("local judge loaded:", JUDGE_ID)
    except Exception as e:
        print("local judge unavailable:", str(e)[:120]); USE_LOCAL_JUDGE = False

@torch.inference_mode()
def judge_batch(items, batch=4):
    """items: list of (article, gold, pred, reason). Returns list of (score, why)."""
    out = []
    for i in range(0, len(items), batch):
        chunk = items[i:i+batch]
        prompts = [judge_tok.apply_chat_template(judge_prompt(*c), tokenize=False,
                                                 add_generation_prompt=True) for c in chunk]
        enc = judge_tok(prompts, return_tensors="pt", padding=True,
                        truncation=True, max_length=768).to("cuda")
        g = judge_model.generate(**enc, max_new_tokens=80, do_sample=False,  # temp 0
                                 pad_token_id=judge_tok.pad_token_id)
        txts = judge_tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        out += [parse_judge(t) for t in txts]
        print("  judging %d/%d" % (min(i+batch,len(items)), len(items)), end="\r")
    print()
    return out


In [ ]:
# ---- Judge backend B: API (what you would actually ship) --------------------
# Uncomment and add a key to use. Kept here so you know the real shape of it.
#
# import os, requests
# def api_judge(article, gold, pred, reason, model="claude-sonnet-4-6"):
#     r = requests.post("https://api.anthropic.com/v1/messages",
#         headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"],
#                  "anthropic-version": "2023-06-01", "content-type":"application/json"},
#         json={"model": model, "max_tokens": 200, "temperature": 0,
#               "system": JUDGE_RUBRIC,
#               "messages": [judge_prompt(article,gold,pred,reason)[1]]})
#     return parse_judge(r.json()["content"][0]["text"])
#
# Production notes:
#   * cache judgements keyed by hash(prompt+model+version) -- judging is the
#     expensive part of CI and inputs repeat constantly
#   * PIN the judge model version. A silent provider upgrade shifts your metric
#     and you will spend a day debugging a "regression" that never happened
#   * batch + retry with exponential backoff on 429/529
print("API judge template above (commented).")


In [ ]:
N_JUDGE = 60      # judging is slow; a stratified subsample is standard practice
idx = list(range(min(N_JUDGE, len(test_rows))))

def to_items(recs):
    return [(recs[i]["text"], recs[i]["gold"], recs[i]["pred"] or "NONE", recs[i]["reason"]) for i in idx]

if USE_LOCAL_JUDGE:
    print("Judging BASE...");  jb = judge_batch(to_items(base_recs))
    print("Judging TUNED..."); jt = judge_batch(to_items(tuned_recs))
else:
    rng = np.random.default_rng(SEED)     # simulated fallback so the notebook still runs
    jb = [(int(np.clip(rng.normal(2.9,1.0),1,5)),"sim") for _ in idx]
    jt = [(int(np.clip(rng.normal(4.0,0.8),1,5)),"sim") for _ in idx]

sb_ = [s for s,_ in jb if s]; st_ = [s for s,_ in jt if s]
print("\nBASE  judge mean: %.2f  (n=%d)" % (np.mean(sb_), len(sb_)))
print("TUNED judge mean: %.2f  (n=%d)" % (np.mean(st_), len(st_)))

fig, ax = plt.subplots(figsize=(8,4))
w=0.38; xs=np.arange(1,6)
ax.bar(xs-w/2, [sb_.count(k) for k in xs], w, label="base",  color="tab:red",  alpha=.75)
ax.bar(xs+w/2, [st_.count(k) for k in xs], w, label="tuned", color="tab:green",alpha=.75)
ax.set_xlabel("judge score"); ax.set_ylabel("count"); ax.set_xticks(xs)
ax.set_title("Tier 2: LLM-as-judge score distribution"); ax.legend(); plt.tight_layout(); plt.show()

print("""If the distribution is bunched at one value, your judge has LENIENCY BIAS
and the rubric anchors are not discriminative enough. Fix the rubric before
you trust the mean. A judge that gives everything a 4 measures nothing.""")


In [ ]:
# --- Position-bias check (pairwise mode) ---
# Present the same two rationales in both orders. If the judge is unbiased the
# verdicts should mirror. The SWAP RATE is your bias measurement.
PAIRWISE = """Compare two rationales for the same article. Which better justifies the correct topic?
Return ONLY JSON: {"winner": "A"} or {"winner": "B"} or {"winner": "tie"}"""

@torch.inference_mode()
def pairwise(article, gold, ra, rb):
    def ask(x, y):
        msgs=[{"role":"system","content":PAIRWISE},
              {"role":"user","content":"ARTICLE: %s\nCORRECT TOPIC: %s\n\nA: %s\n\nB: %s" % (article[:250], gold, x, y)}]
        p = judge_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        e = judge_tok(p, return_tensors="pt").to("cuda")
        o = judge_model.generate(**e, max_new_tokens=24, do_sample=False,
                                 pad_token_id=judge_tok.pad_token_id)
        t = judge_tok.decode(o[0, e["input_ids"].shape[1]:], skip_special_tokens=True)
        m = re.search(r'"winner"\s*:\s*"(A|B|tie)"', t)
        return m.group(1) if m else "tie"
    return ask(ra, rb), ask(rb, ra)   # forward and reversed

if USE_LOCAL_JUDGE:
    agree = swap = 0
    for i in idx[:20]:
        f, rv = pairwise(base_recs[i]["text"], base_recs[i]["gold"],
                         base_recs[i]["reason"], tuned_recs[i]["reason"])
        consistent = (f=="A" and rv=="B") or (f=="B" and rv=="A") or (f=="tie" and rv=="tie")
        agree += consistent; swap += (not consistent)
    print("order-consistent verdicts: %d/20" % agree)
    print("POSITION BIAS RATE       : %.0f%%" % (100*swap/20))
    print("""
Published position-bias rates for strong judges run roughly 10-25%. If yours is
near 50%, the judge is essentially guessing. The standard mitigation is exactly
what we did: evaluate BOTH orders and count only order-consistent verdicts.""")
else:
    print("(skipped - no local judge)")


---
# 12. TIER 3 — Human agreement via Cohen's kappa

**The tier that makes the other two trustworthy.** You label a sample by hand, then measure how well the judge agrees with you.

### Why raw agreement is not enough

Suppose you and the judge both label "acceptable" 90% of the time. Even if you were both flipping biased coins independently, you'd agree ~82% of the time **by chance alone**. Reporting "82% agreement" would be a lie.

### Cohen's kappa corrects for chance

$$ \kappa = \frac{p_o - p_e}{1 - p_e} $$

- `p_o` = observed agreement.
- `p_e` = agreement expected by chance, computed from each rater's **marginal** distribution: `p_e = Σ_k P₁(k)·P₂(k)`.
- Numerator = agreement above chance. Denominator = the maximum possible agreement above chance.
- `κ = 1` perfect; `κ = 0` no better than chance; `κ < 0` **worse** than chance (systematic disagreement — usually means a label-mapping bug).

### Landis & Koch interpretation
| κ | reading |
|---|---|
| < 0.00 | poor |
| 0.00–0.20 | slight |
| 0.21–0.40 | fair |
| 0.41–0.60 | moderate |
| 0.61–0.80 | **substantial** ← the usual bar for trusting a judge |
| 0.81–1.00 | almost perfect |

### Weighted kappa for ordinal scales
Our judge outputs 1–5, which is **ordinal**: predicting 4 when the human said 5 is a much smaller error than predicting 1. Plain kappa treats both as equally wrong. Use **quadratic weighted kappa** for ordinal scales — it penalizes by squared distance. (Note: quadratic weighted kappa is algebraically equivalent to an intraclass correlation, which is why it's the standard for ordinal rating agreement.)

### Related measures worth naming
- **Fleiss' kappa** — 3+ raters.
- **Krippendorff's alpha** — any number of raters, handles missing data and any measurement level. The most general choice.
- **Cohen's kappa** — exactly 2 raters. Ours.

### The paradox to know about
Kappa is **sensitive to prevalence**. With very skewed marginals you can get high `p_o` but low `κ`. If that happens, report `p_o`, `κ`, **and** the marginal distributions — don't just report κ and let someone draw the wrong conclusion.


In [ ]:
# ---- Cohen's kappa from scratch (be able to do this on a whiteboard) -------
def cohens_kappa(a, b, weights=None):
    cats = sorted(set(a) | set(b))
    n = len(a); K = len(cats)
    ci = {c:i for i,c in enumerate(cats)}
    O = np.zeros((K,K))
    for x,y in zip(a,b): O[ci[x], ci[y]] += 1
    O /= n
    r = O.sum(1); c = O.sum(0)          # marginals of each rater
    E = np.outer(r, c)                  # expected agreement under independence

    if weights is None:                 # unweighted: disagreement costs 1
        W = 1 - np.eye(K)
    elif weights == "linear":
        W = np.abs(np.arange(K)[:,None] - np.arange(K)[None,:]) / (K-1)
    elif weights == "quadratic":
        W = (np.arange(K)[:,None] - np.arange(K)[None,:])**2 / (K-1)**2
    po = 1 - (W*O).sum()
    pe = 1 - (W*E).sum()
    return (po-pe)/(1-pe) if (1-pe) > 1e-12 else 0.0, po, pe

def interpret(k):
    for t,l in [(0.0,"POOR"),(0.20,"slight"),(0.40,"fair"),(0.60,"moderate"),
                (0.80,"SUBSTANTIAL"),(1.01,"almost perfect")]:
        if k < t: return l
    return "almost perfect"

# Sanity checks against sklearn
from sklearn.metrics import cohen_kappa_score
rng = np.random.default_rng(0)
x = rng.integers(1,6,300); y = x.copy()
print("identical raters      : kappa=%.3f (expect 1.0)" % cohens_kappa(x,y)[0])
y2 = rng.integers(1,6,300)
print("independent raters    : kappa=%.3f (expect ~0)" % cohens_kappa(x,y2)[0])
print("matches sklearn       :", np.isclose(cohens_kappa(x,y2)[0], cohen_kappa_score(x,y2)))

# The chance-correction demo -- the number that makes the concept click
a3 = ["ok"]*90 + ["bad"]*10
b3 = ["ok"]*88 + ["bad"]*2 + ["ok"]*2 + ["bad"]*8
k3, po3, pe3 = cohens_kappa(a3, b3)
print("\nSkewed example: observed agreement %.2f, but CHANCE agreement is %.2f" % (po3, pe3))
print("               -> kappa = %.3f  (%s)" % (k3, interpret(k3)))
print("This is why '90%% agreement!' means nothing without the marginals.")


In [ ]:
# ---- Collect "human" labels and measure judge validity ---------------------
# In reality: export a stratified sample to a spreadsheet, have 2 people label
# with a written guideline, measure human-human kappa FIRST (that is your
# ceiling -- if humans only agree at 0.5, no judge can beat 0.5), then
# adjudicate disagreements into a gold set.
#
# Here we simulate a human who agrees with the judge most of the time but has
# a systematic stricter tendency -- a realistic pattern.
rng = np.random.default_rng(SEED)
def simulate_human(judge_scores, noise=0.75, bias=-0.25):
    out=[]
    for s in judge_scores:
        v = s + rng.normal(bias, noise)
        out.append(int(np.clip(round(v),1,5)))
    return out

judge_scores = [s for s,_ in jt if s]
human_scores = simulate_human(judge_scores)

k_un, po, pe = cohens_kappa(human_scores, judge_scores)
k_lin,_,_    = cohens_kappa(human_scores, judge_scores, weights="linear")
k_quad,_,_   = cohens_kappa(human_scores, judge_scores, weights="quadratic")

print("n = %d rated items\n" % len(judge_scores))
print("observed agreement (exact)   : %.3f" % po)
print("chance agreement             : %.3f" % pe)
print("Cohen's kappa (unweighted)   : %.3f  -> %s" % (k_un,   interpret(k_un)))
print("kappa (linear weights)       : %.3f  -> %s" % (k_lin,  interpret(k_lin)))
print("kappa (QUADRATIC - use this) : %.3f  -> %s" % (k_quad, interpret(k_quad)))
print("""
Quadratic weighting is the right choice for an ordinal 1-5 rubric: it says a
4-vs-5 disagreement is minor and a 1-vs-5 disagreement is severe. Unweighted
kappa treats them identically, which understates a judge that is 'close but
not exact' -- exactly the common case.""")

# Also collapse to a binary decision, since that is what a CI gate acts on
h_bin = [1 if s>=4 else 0 for s in human_scores]
j_bin = [1 if s>=4 else 0 for s in judge_scores]
kb,pob,peb = cohens_kappa(h_bin, j_bin)
print("\nBinary (acceptable >= 4): kappa = %.3f  (%s), raw agreement %.3f" % (kb, interpret(kb), pob))
print("Gate on the BINARY decision, and validate the BINARY kappa -- that is")
print("the actual decision your CI makes, so that is what must be trustworthy.")

# Agreement matrix
cm = np.zeros((5,5), int)
for h,j in zip(human_scores, judge_scores): cm[h-1, j-1]+=1
fig,ax=plt.subplots(figsize=(5.5,4.5))
ax.imshow(cm, cmap="Purples")
ax.set_xticks(range(5)); ax.set_xticklabels(range(1,6)); ax.set_xlabel("judge score")
ax.set_yticks(range(5)); ax.set_yticklabels(range(1,6)); ax.set_ylabel("human score")
ax.set_title("Human vs judge (κ_quad = %.2f)" % k_quad)
for i in range(5):
    for j in range(5):
        if cm[i,j]: ax.text(j,i,cm[i,j],ha="center",va="center")
plt.tight_layout(); plt.show()


---
# 13. McNemar's test — the correct test for comparing two models

### The setup
Two models, **same test set**, binary outcome per item (correct / incorrect). Build the 2×2 contingency table of **paired** outcomes:

|  | Tuned ✓ | Tuned ✗ |
|---|---|---|
| **Base ✓** | a | b |
| **Base ✗** | c | d |

### The key insight
`a` (both right) and `d` (both wrong) carry **no information about which model is better**. Only the **discordant** pairs matter:
- `b` = base right, tuned wrong (tuned broke something)
- `c` = base wrong, tuned right (tuned fixed something)

Under H₀ (the models are equally good), each discordant item is a coin flip: `b ~ Binomial(b+c, 0.5)`.

$$ \chi^2 = \frac{(|b-c|-1)^2}{b+c}, \quad df = 1 $$

The `−1` is Yates' continuity correction (a continuous χ² approximating a discrete binomial). **When `b+c < 25`, don't use the χ² approximation at all — use the exact binomial test.** Knowing that threshold is a classic interview differentiator.

### Why not a t-test or a two-proportion z-test?
Because the samples are **not independent** — they are the *same items* scored twice. Both models find the same easy items easy. A paired design has far lower variance, so an unpaired test throws away power and gives a p-value that is too conservative. Using an independent test on paired data is a textbook statistical error, and it's one interviewers actively probe for.

### Reporting properly
Always report **effect size alongside significance**. With n=10,000 a 0.2% difference is "significant" and irrelevant. State: accuracy delta, the discordant counts `b`/`c`, the p-value, **and** a confidence interval (Section 14).

> **The line to remember:** "Significance tells you the difference is real. Effect size tells you whether it's worth shipping. You need both."


In [ ]:
# ---- McNemar from scratch --------------------------------------------------
from scipy import stats as sps

def mcnemar(base_correct, tuned_correct, exact_threshold=25):
    a=b=c=d=0
    for x,y in zip(base_correct, tuned_correct):
        if   x==1 and y==1: a+=1
        elif x==1 and y==0: b+=1     # tuned BROKE it
        elif x==0 and y==1: c+=1     # tuned FIXED it
        else:               d+=1
    n_disc = b+c
    if n_disc == 0:
        return {"a":a,"b":b,"c":c,"d":d,"n_discordant":0,"p_value":1.0,
                "statistic":0.0,"method":"no discordant pairs"}
    if n_disc < exact_threshold:
        # Exact binomial: P(X >= max(b,c)) two-sided under p=0.5
        p = sps.binomtest(max(b,c), n_disc, 0.5, alternative="two-sided").pvalue
        return {"a":a,"b":b,"c":c,"d":d,"n_discordant":n_disc,"p_value":p,
                "statistic":float(max(b,c)),"method":"exact binomial (b+c<25)"}
    chi2 = (abs(b-c)-1)**2 / n_disc         # Yates-corrected
    p = 1 - sps.chi2.cdf(chi2, df=1)
    return {"a":a,"b":b,"c":c,"d":d,"n_discordant":n_disc,"p_value":p,
            "statistic":chi2,"method":"chi-square with continuity correction"}

bc = [r["correct"] for r in base_recs]
tc = [r["correct"] for r in tuned_recs]
res = mcnemar(bc, tc)

print("Contingency table (paired outcomes on the SAME %d items)\n" % len(bc))
print("                 tuned OK   tuned WRONG")
print("  base OK      %9d %13d" % (res["a"], res["b"]))
print("  base WRONG   %9d %13d" % (res["c"], res["d"]))
print("\nDiscordant pairs : %d  (b=%d broke, c=%d fixed)" % (res["n_discordant"], res["b"], res["c"]))
print("Method           : %s" % res["method"])
print("Statistic        : %.4f" % res["statistic"])
print("p-value          : %.5f" % res["p_value"])
print("Verdict (a=0.05) : %s" % ("SIGNIFICANT - the improvement is real"
                                 if res["p_value"]<0.05 else
                                 "NOT significant - could be noise"))
print("\nAccuracy delta   : %+.4f  (%.4f -> %.4f)" % (np.mean(tc)-np.mean(bc), np.mean(bc), np.mean(tc)))

# Cross-check against statsmodels
try:
    from statsmodels.stats.contingency_tables import mcnemar as sm_mcnemar
    tbl = [[res["a"],res["b"]],[res["c"],res["d"]]]
    sm = sm_mcnemar(tbl, exact=res["n_discordant"]<25, correction=True)
    print("\nstatsmodels cross-check: p = %.5f" % sm.pvalue)
except Exception as e:
    print("\n(statsmodels unavailable)", str(e)[:60])


In [ ]:
# ---- Why the paired test matters: same data, two tests ---------------------
n = len(bc)
p1, p2 = np.mean(bc), np.mean(tc)

# WRONG: two-proportion z-test, which assumes independent samples
pool = (p1*n + p2*n)/(2*n)
se = np.sqrt(pool*(1-pool)*(2/n))
z = (p2-p1)/se if se>0 else 0
p_indep = 2*(1-sps.norm.cdf(abs(z)))

print("Two-proportion z-test (WRONG - assumes independence) : p = %.5f" % p_indep)
print("McNemar's test        (CORRECT - paired)             : p = %.5f" % res["p_value"])
print("""
The paired test is almost always more powerful because it removes the variance
contributed by item difficulty. Both models get the easy items right; the paired
test simply ignores those and asks only about the items where they DISAGREE.

Say this in an interview: 'Same test set means paired data means McNemar, not
a z-test or a t-test.'""")

# Inspect what actually changed -- more useful than any p-value
fixed  = [i for i,(x,y) in enumerate(zip(bc,tc)) if x==0 and y==1]
broken = [i for i,(x,y) in enumerate(zip(bc,tc)) if x==1 and y==0]
print("\n--- REGRESSIONS (fine-tuning broke these) ---")
for i in broken[:3]:
    print("  gold=%-9s base=%-9s tuned=%-9s | %s" % (
        base_recs[i]["gold"], base_recs[i]["pred"], tuned_recs[i]["pred"], base_recs[i]["text"][:80]))
print("\n--- FIXES ---")
for i in fixed[:3]:
    print("  gold=%-9s base=%-9s tuned=%-9s | %s" % (
        base_recs[i]["gold"], base_recs[i]["pred"], tuned_recs[i]["pred"], base_recs[i]["text"][:80]))
print("""
ALWAYS look at the regressions, not just the net delta. A model that gains 5%
overall while breaking a category your biggest customer depends on is a
shipping decision, not a metric.""")


---
# 14. Bootstrap confidence intervals

A point estimate without uncertainty is not a result. **Bootstrap** is the general-purpose way to get a CI for any statistic without distributional assumptions.

**Procedure:** resample your test set *with replacement* B times (B ≈ 2000–10000), recompute the metric each time, take the 2.5th and 97.5th percentiles.

For a **paired** comparison (which is what we have), resample **indices**, then compute both models' metrics on those same indices — preserving the pairing. That gives you a CI on the **delta**, which is the number you actually care about.

**How to read it:** if the 95% CI on the delta excludes 0, the improvement is significant at α=0.05 — consistent with McNemar. The CI is more useful than the p-value because it also tells you the plausible *magnitude*.

> A 200-item test set gives roughly ±7% CI width on an accuracy near 0.5. If your gate threshold is tighter than your CI, **your gate is measuring noise.** Compute the CI first, then set the threshold outside it. This single check prevents a whole class of flaky-CI misery.


In [ ]:
def bootstrap_delta(bc, tc, B=5000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    bc = np.array(bc); tc = np.array(tc); n = len(bc)
    deltas = np.empty(B); b_acc = np.empty(B); t_acc = np.empty(B)
    for i in range(B):
        idx = rng.integers(0, n, n)     # SAME indices for both -> pairing preserved
        b_acc[i] = bc[idx].mean(); t_acc[i] = tc[idx].mean()
        deltas[i] = t_acc[i] - b_acc[i]
    lo, hi = np.percentile(deltas, [100*alpha/2, 100*(1-alpha/2)])
    return {"delta": tc.mean()-bc.mean(), "ci": (lo,hi),
            "base_ci": tuple(np.percentile(b_acc,[2.5,97.5])),
            "tuned_ci": tuple(np.percentile(t_acc,[2.5,97.5])),
            "p_boot": float(np.mean(deltas <= 0)*2), "deltas": deltas}

bs = bootstrap_delta(bc, tc)
print("base  accuracy : %.4f  95%% CI [%.4f, %.4f]" % (np.mean(bc), *bs["base_ci"]))
print("tuned accuracy : %.4f  95%% CI [%.4f, %.4f]" % (np.mean(tc), *bs["tuned_ci"]))
print("DELTA          : %+.4f  95%% CI [%+.4f, %+.4f]" % (bs["delta"], *bs["ci"]))
print("CI excludes 0  : %s" % (bs["ci"][0] > 0 or bs["ci"][1] < 0))
print("bootstrap p    : %.4f   (McNemar p: %.4f)" % (min(bs["p_boot"],1.0), res["p_value"]))

plt.figure(figsize=(9,3.8))
plt.hist(bs["deltas"], bins=60, color="tab:blue", alpha=.75)
plt.axvline(0, color="k", ls="--", label="no change")
plt.axvline(bs["ci"][0], color="r", ls=":"); plt.axvline(bs["ci"][1], color="r", ls=":", label="95% CI")
plt.axvline(bs["delta"], color="tab:green", lw=2, label="observed")
plt.xlabel("accuracy delta (tuned - base)"); plt.ylabel("bootstrap draws")
plt.title("Paired bootstrap: uncertainty on the improvement"); plt.legend()
plt.tight_layout(); plt.show()

width = bs["ci"][1]-bs["ci"][0]
print("\nCI width = %.3f on n=%d items." % (width, len(bc)))
print("Rule of thumb: CI width shrinks as 1/sqrt(n). To HALVE it, QUADRUPLE the test set.")
print("Set your CI gate threshold WIDER than this, or you will block good PRs on noise.")


---
# 15. The eval CI gate

> **The gate is what turns an evaluation into engineering.** Without it, evals are a report someone reads once. With it, they are a constraint the system cannot violate.

### Design principles

1. **Absolute floors AND regression deltas.** "Accuracy ≥ 0.80" catches bad models; "accuracy ≥ previous − 0.02" catches slow decay that never trips the floor.
2. **Gate on what breaks production.** Format compliance and unparseable rate often deserve *harder* thresholds than accuracy. A parse failure is an outage; a 1% accuracy dip is a Tuesday.
3. **Thresholds must sit outside your noise band.** Compute the bootstrap CI first (Section 14).
4. **Judge validity is itself a gate.** If judge-vs-human κ drops below 0.6, the Tier-2 metric is no longer trustworthy — fail the build on *that*, not on the score it produced.
5. **Tiered execution for cost.** Tier 1 on every commit (seconds, free). Tier 2 on PRs to main (minutes, costs API money). Tier 3 quarterly or when the judge model version changes.
6. **Pin everything.** Judge model version, dataset revision, seed, decoding params. An unpinned judge is a metric that changes under you.
7. **Non-zero exit code.** That's the entire interface between your harness and CI.

### The seventh point deserves emphasis
Everything above is just: *write a script that returns 1 when quality drops.* CI systems understand exit codes and nothing else. Whatever your evaluation philosophy, it has to end in `sys.exit(1)`.


In [ ]:
GATE_CONFIG = {
    "absolute": {          # hard floors -- never ship below these
        "accuracy":     0.75,
        "macro_f1":     0.70,
        "format_ok":    0.95,   # stricter than accuracy: parse failures = outage
        "judge_mean":   3.5,
    },
    "regression": {        # max allowed drop vs the previous release
        "accuracy":    -0.02,
        "macro_f1":    -0.03,
        "format_ok":   -0.01,
        "judge_mean":  -0.20,
    },
    "validity": {          # is the measurement instrument still trustworthy?
        "judge_human_kappa_min": 0.60,
        "min_eval_items": 100,
    },
}

def eval_gate(current, previous, judge_kappa, n_items, cfg=GATE_CONFIG, verbose=True):
    failures, warnings_ = [], []

    if n_items < cfg["validity"]["min_eval_items"]:
        failures.append("eval set too small: %d < %d (CIs will be uselessly wide)"
                        % (n_items, cfg["validity"]["min_eval_items"]))
    if judge_kappa is not None and judge_kappa < cfg["validity"]["judge_human_kappa_min"]:
        failures.append("JUDGE INVALID: kappa %.3f < %.2f -- Tier 2 numbers cannot be trusted"
                        % (judge_kappa, cfg["validity"]["judge_human_kappa_min"]))

    for k, floor in cfg["absolute"].items():
        if k in current and current[k] < floor:
            failures.append("floor: %s = %.3f < %.3f" % (k, current[k], floor))

    if previous:
        for k, max_drop in cfg["regression"].items():
            if k in current and k in previous:
                d = current[k]-previous[k]
                if d < max_drop:
                    failures.append("REGRESSION: %s %+.3f (limit %+.3f)" % (k, d, max_drop))
                elif d < 0:
                    warnings_.append("%s slipped %+.3f (within tolerance)" % (k, d))

    if verbose:
        print("="*62); print("EVAL CI GATE"); print("="*62)
        for k in sorted(set(list(cfg['absolute'])+list(cfg['regression']))):
            if k in current:
                prev = previous.get(k) if previous else None
                print("  %-12s %.3f %s" % (k, current[k],
                      ("(prev %.3f, %+.3f)" % (prev, current[k]-prev)) if prev is not None else ""))
        print("  judge kappa  %s" % ("%.3f" % judge_kappa if judge_kappa is not None else "n/a"))
        print("-"*62)
        for w in warnings_: print("  WARN  " + w)
        for f in failures: print("  FAIL  " + f)
        print("-"*62)
        print("  RESULT: " + ("PASS - safe to ship" if not failures else "BLOCKED (%d failures)" % len(failures)))
        print("="*62)
    return len(failures)==0, failures, warnings_

prev_metrics = {"accuracy": t1b["accuracy"], "macro_f1": t1b["macro_f1"],
                "format_ok": sb["format_ok"], "judge_mean": float(np.mean(sb_))}
cur_metrics  = {"accuracy": t1t["accuracy"], "macro_f1": t1t["macro_f1"],
                "format_ok": st["format_ok"], "judge_mean": float(np.mean(st_))}

passed, fails, warns = eval_gate(cur_metrics, prev_metrics, k_quad, len(test_rows))


In [ ]:
# ---- Write the actual artifacts you would commit ---------------------------
report = {
    "model": BASE_ID, "adapter": "qlora-adapter", "seed": SEED,
    "n_test": len(test_rows), "n_train": len(train_rows),
    "tier1": {"base": {k: t1b[k] for k in ["accuracy","macro_f1","micro_f1"]},
              "tuned": {k: t1t[k] for k in ["accuracy","macro_f1","micro_f1"]},
              "format_ok_base": sb["format_ok"], "format_ok_tuned": st["format_ok"]},
    "tier2": {"judge_model": JUDGE_ID if USE_LOCAL_JUDGE else "simulated",
              "base_mean": float(np.mean(sb_)), "tuned_mean": float(np.mean(st_)),
              "n_judged": len(st_)},
    "tier3": {"kappa_unweighted": k_un, "kappa_quadratic": k_quad,
              "kappa_binary": kb, "interpretation": interpret(k_quad)},
    "statistics": {"mcnemar_p": res["p_value"], "mcnemar_method": res["method"],
                   "b_broke": res["b"], "c_fixed": res["c"],
                   "delta": bs["delta"], "delta_ci95": list(bs["ci"])},
    "gate": {"passed": passed, "failures": fails, "warnings": warns},
}
with open("eval_report.json","w") as f: json.dump(report, f, indent=2)
print(json.dumps(report, indent=2)[:1400])

# The gate script that CI actually runs
gate_script = '''#!/usr/bin/env python3
# CI entrypoint. Exit 1 blocks the merge.
import json, sys

cur  = json.load(open("eval_report.json"))
try:
    prev = json.load(open("baseline_metrics.json"))
except FileNotFoundError:
    prev = None
    print("No baseline found - absolute floors only (first run).")

FLOORS      = {"accuracy": 0.75, "macro_f1": 0.70, "format_ok": 0.95}
MAX_DROP    = {"accuracy": -0.02, "macro_f1": -0.03, "format_ok": -0.01}
KAPPA_FLOOR = 0.60

m = dict(cur["tier1"]["tuned"]); m["format_ok"] = cur["tier1"]["format_ok_tuned"]
fails = []

if cur["tier3"]["kappa_quadratic"] < KAPPA_FLOOR:
    fails.append("judge validity: kappa %.3f < %.2f" % (cur["tier3"]["kappa_quadratic"], KAPPA_FLOOR))

for k, floor in FLOORS.items():
    if k in m and m[k] < floor:
        fails.append("%s %.4f below floor %.2f" % (k, m[k], floor))

if prev:
    for k, lim in MAX_DROP.items():
        if k in m and k in prev:
            d = m[k] - prev[k]
            if d < lim:
                fails.append("%s regressed %+.4f (limit %+.4f)" % (k, d, lim))

# Effect size sanity: do not celebrate a change inside the noise band
lo, hi = cur["statistics"]["delta_ci95"]
if lo <= 0 <= hi:
    print("NOTE: accuracy delta CI [%.3f, %.3f] includes 0 - change not distinguishable from noise." % (lo, hi))

for f in fails:
    print("FAIL:", f)
print("GATE:", "BLOCKED" if fails else "PASSED")
sys.exit(1 if fails else 0)
'''
open("run_eval_gate.py","w").write(gate_script)

workflow = '''name: model-eval-gate
on:
  pull_request:
    paths: ["models/**", "prompts/**", "training/**"]

jobs:
  tier1:                       # fast, free, every commit
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: "3.11"}
      - run: pip install -r requirements.txt
      - run: python eval/run_tier1.py --out eval_report.json
      - run: python run_eval_gate.py
      - uses: actions/upload-artifact@v4
        if: always()
        with: {name: eval-report, path: eval_report.json}

  tier2:                       # costs money; only on PRs to main
    needs: tier1
    if: github.base_ref == 'main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: pip install -r requirements.txt
      - env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
          JUDGE_MODEL: claude-sonnet-4-6      # PINNED. Never float this.
        run: python eval/run_tier2_judge.py --n 200 --cache .judge_cache
      - run: python run_eval_gate.py
'''
open("eval_gate_workflow.yml","w").write(workflow)
print("\nWrote: eval_report.json, run_eval_gate.py, eval_gate_workflow.yml")


---
# 16. Failure modes — what actually goes wrong

### 1. Test-set contamination
Your eval data leaked into pretraining or into your training split. Signs: suspiciously high scores, perfect performance on old benchmarks but poor real-world behaviour. **Mitigations:** n-gram overlap check between train and test; hold out data **by time** (test on the newest data); build a small private eval set that never leaves your org.

### 2. Overfitting to the eval set
Not the model overfitting — *you* overfitting, by tweaking prompts and hyperparameters against the same test set for weeks. After 50 iterations that test set is effectively training data. **Mitigation:** three splits — dev (iterate freely), test (occasional), and a **locked holdout** touched only before release.

### 3. Judge drift
The provider silently upgrades the judge model and your metric shifts 3 points overnight. **Mitigation:** pin version strings; keep a fixed set of "golden" judged examples and re-judge them on every run — if those scores move, the *judge* changed, not your model.

### 4. Goodhart's law
"When a measure becomes a target, it ceases to be a good measure." Optimizing hard against an LLM judge teaches your model the judge's biases — verbosity, confident tone, structural tics. **Mitigation:** rotate judges, keep human spot-checks, and always carry at least one deterministic metric the judge cannot influence.

### 5. Catastrophic forgetting
Task accuracy up, everything else down. **Mitigation:** always run a general-capability eval alongside your task eval. This is a mandatory second gate, not optional.

### 6. Distribution shift
Your eval set is from March; production traffic in September looks different. **Mitigation:** refresh eval data from real traffic on a schedule; monitor production input distribution, not just outputs.

### 7. The silent one: aggregate-only logging
If you only save accuracy, you can never run McNemar, never bootstrap, never inspect regressions. **Always persist per-item predictions.** It costs a few MB and it's the difference between having an eval harness and having a number.


In [ ]:
# Contamination check: n-gram overlap between train and test
def ngrams(s, n=8):
    w = s.lower().split()
    return set(tuple(w[i:i+n]) for i in range(max(0,len(w)-n+1)))

train_ng = set()
for r in train_rows: train_ng |= ngrams(r["text"])
hits = [r for r in test_rows if ngrams(r["text"]) & train_ng]
print("CONTAMINATION CHECK (8-gram overlap)")
print("  test items sharing an 8-gram with train: %d / %d (%.1f%%)"
      % (len(hits), len(test_rows), 100*len(hits)/len(test_rows)))
print("  %s" % ("CLEAN" if len(hits)/len(test_rows) < 0.02 else "INVESTIGATE - possible leakage"))

# Sanity check that the harness itself is not broken:
# a random-guessing model must score ~1/num_classes.
rng = np.random.default_rng(0)
rand_correct = [int(LABELS[rng.integers(0,4)] == r["label"]) for r in test_rows]
print("\nHARNESS SANITY: random baseline accuracy = %.3f (expect ~%.3f)"
      % (np.mean(rand_correct), 1/len(LABELS)))
print("If a random model scores well above chance, your PARSER is leaking the answer.")
print("Always run this check. It catches harness bugs that look like model quality.")


---
# 17. Interview cheat sheet

### QLoRA
**Q: What are the three components of QLoRA?**
NF4 4-bit quantization, double quantization (quantizing the block-wise scaling constants), paged optimizers (unified-memory paging of optimizer state to avoid OOM spikes).

**Q: Why NF4 rather than INT4?**
Weights are approximately normally distributed. NF4 places its 16 levels at the quantiles of a normal so each holds equal probability mass — information-theoretically optimal for that distribution. INT4's uniform levels waste resolution in the tails.

**Q: Does QLoRA make training faster?**
No — slightly *slower*, because weights are dequantized on the fly for every matmul. It saves **memory**, not time. It trades compute for VRAM.

**Q: Where does the memory saving mostly come from?**
Not just the 4× on base weights — mainly from having **no gradients and no Adam optimizer state** for the frozen base. Adam alone is 2 fp32 moments per trainable parameter.

### LoRA
**Q: Why initialize B to zero?**
So `BA = 0` at step 0 and the model is exactly the pretrained model. Training starts from a known-good point instead of injecting random noise into every layer.

**Q: What does alpha do?**
Scales the update by `α/r`, decoupling the effective learning rate from the rank so you don't retune LR when you change `r`. Convention `α = 2r`.

**Q: Which modules do you target?**
All linear layers, including the MLP (`gate/up/down`), not just attention — the QLoRA paper found this matters substantially.

**Q: Why is the LoRA learning rate ~2e-4, not 2e-5?**
Few, freshly-initialized parameters with a frozen base. Standard full-FT learning rates are far too small and produce a flat loss curve.

### Evaluation
**Q: Why three tiers?**
Deterministic metrics are cheap and trustworthy but can't judge open-ended text. LLM judges scale to open-ended text but are biased instruments. Human agreement calibrates the instrument. Each tier covers the previous tier's blind spot.

**Q: Why Cohen's kappa instead of raw agreement?**
Raw agreement is inflated by chance, especially with skewed marginals. Kappa subtracts expected chance agreement: `(p_o − p_e)/(1 − p_e)`.

**Q: Which kappa for a 1–5 rubric?**
**Quadratic weighted** — the scale is ordinal, so a 4-vs-5 disagreement should count far less than 1-vs-5.

**Q: Why McNemar and not a t-test?**
Same test set = paired data. McNemar conditions on the discordant pairs (`b`, `c`) and discards items both models got right or wrong, which is where the information is. An independent test assumes independence you don't have, and loses power.

**Q: When do you use the exact version of McNemar?**
When `b + c < 25`; the χ² approximation is unreliable there. Use the exact binomial test instead.

**Q: What are the biggest LLM-judge biases and how do you handle them?**
Position (evaluate both orders, report swap rate), verbosity (length-controlled rubric), self-preference (use a different model family), leniency (anchored rubric descriptors). And validate against humans with kappa.

**Q: How do you set CI gate thresholds?**
Bootstrap the metric to get its noise band first, then set the threshold outside it. A threshold inside your confidence interval produces flaky builds and teaches the team to ignore the gate.

**Q: What do you gate on besides accuracy?**
Format compliance, unparseable rate, latency/output length, a general-capability eval (catastrophic forgetting), and **judge validity itself** (kappa floor).

---

## The one-breath project summary

> "I fine-tuned a 0.5B model with QLoRA — NF4 base, LoRA adapters on all linear layers, r=16 — and built a three-tier eval harness around it: deterministic metrics, an LLM judge with an anchored rubric and position-bias controls, and human calibration measured with quadratic weighted Cohen's kappa. Model comparison used McNemar's paired test with bootstrap CIs on the delta, and the whole thing runs as a CI gate that blocks merges on regressions — including a gate on the *judge's* validity, because an uncalibrated judge is a metric you can't trust."

That last clause is the one that gets remembered.
